# SACDpy continuous-video z-stack pipeline

Discover FOV folders below one dataset root, read ONI time/z TIFF movies only from each `pos_0/`, reconstruct every movie, and write one SACD `TZYX` stack plus one SACD z-MIP `TYX` movie per FOV. Outputs are written to the dataset root.

## 1. Imports

In [ ]:
from contextlib import nullcontext
from pathlib import Path
import os
from time import perf_counter
import sys

import matplotlib.pyplot as plt
import numpy as np
import tifffile
from rich.progress import BarColumn, MofNCompleteColumn, Progress, SpinnerColumn, TextColumn, TimeElapsedColumn, TimeRemainingColumn

repo = Path.cwd()
src = repo / 'src'
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

from sacdpy import SACDParams, read_tiff_stack, reconstruct
from sacdpy.discrete_timelapse import (
    apply_incomplete_grid_policy, discover_folder_plans, discover_fov_folders,
    infer_na, infer_pixel_nm, infer_time_interval_s, infer_z_spacing_um,
    output_paths_for_group, validate_unique_output_paths, write_timelapse_tiff,
)
print(f'Working folder: {repo}')

## 2. Configuration
Edit the include/exclude patterns before discovery. A pattern may match the full relative FOV path or any path component.

In [ ]:
dataset_root = Path('/Volumes/guttman/primarydata/imaging_rawdata/20260529_ONI-gmgao-SPEN_SACDlive-Aux_Dox_undiff')
position_folder = 'pos_0'
include_folder_patterns = ()
exclude_folder_patterns = ('50ms_photobleached',)
glob_pattern = '*.tif'
channel_name = '647'
incomplete_grid_policy = 'drop_incomplete_timepoints'
overwrite_outputs = False
show_progress = True
run_full_batch = os.environ.get('SACDPY_RUN_FULL_BATCH', '0') == '1'

wavelength_nm = 647.0
pixel_nm = None
fallback_pixel_nm = 117.0
na = None
fallback_na = 1.45
mag, iter1, iter2, ac_order, subfactor = 2, 7, 8, 2, 0.8
ifbackground, backgroundfactor = False, 2.0
ifregistration, ifsparsedecon = False, False
fidelity, tcontinuity, sparsity, sparse_iterations = 100.0, 0.1, 1.0, 100
time_interval_source = 'metadata'
nominal_time_interval_s = None

## 3. Discovery and preflight
This cell is read-only. Review the included/excluded folders, time/z grids, metadata, and planned outputs before reconstruction.

In [ ]:
discovery = discover_fov_folders(
    dataset_root, position_folder=position_folder,
    include_folder_patterns=include_folder_patterns,
    exclude_folder_patterns=exclude_folder_patterns,
)
print(f'Discovered {len(discovery.discovered)} FOV(s); included {len(discovery.included)}; excluded {len(discovery.excluded)}')
print('Included:')
for folder in discovery.included:
    print(' +', folder.relative_to(dataset_root))
print('Excluded:')
for folder in discovery.excluded:
    print(' -', folder.relative_to(dataset_root))
if not discovery.included:
    raise ValueError('No FOV folders remain after filtering.')

raw_plans = discover_folder_plans(
    [folder / position_folder for folder in discovery.included],
    glob_pattern=glob_pattern, channel_name=channel_name, recursive=False, output_dir=dataset_root,
)
prepared = []
for fov_folder, plan in zip(discovery.included, raw_plans):
    if not plan.groups:
        raise FileNotFoundError(f'No matching ONI TIFFs in {fov_folder / position_folder}')
    for original_group in plan.groups:
        grid_result = apply_incomplete_grid_policy(original_group, incomplete_grid_policy)
        if grid_result.group is None:
            print(f'SKIP {fov_folder.name}: {grid_result.status}; dropped={grid_result.dropped_time_indices}')
            continue
        group = grid_result.group
        first_file = next(iter(group.files.values()))
        resolved_pixel_nm = pixel_nm if pixel_nm is not None else infer_pixel_nm(first_file, fallback_pixel_nm)
        resolved_na = na if na is not None else infer_na(first_file, fallback_na)
        z_spacing_um = infer_z_spacing_um(group)
        time_interval_s = infer_time_interval_s(group, source=time_interval_source, nominal_time_interval_s=nominal_time_interval_s)
        stack_output, mip_output = output_paths_for_group(group, dataset_root)
        prepared.append({
            'fov_folder': fov_folder, 'group': group, 'dropped_time_indices': grid_result.dropped_time_indices,
            'grid_status': grid_result.status, 'pixel_nm': float(resolved_pixel_nm), 'na': float(resolved_na),
            'z_spacing_um': z_spacing_um, 'time_interval_s': time_interval_s,
            'stack_output': stack_output, 'mip_output': mip_output,
        })
        print(f'{fov_folder.relative_to(dataset_root)}: files={len(group.files)}, T={len(group.time_indices)}, Z={len(group.z_indices)}, dropped={grid_result.dropped_time_indices}')
        print(f'  pixel_nm={resolved_pixel_nm}, na={resolved_na}, z_spacing_um={z_spacing_um}, dt_s={time_interval_s}')
        print(f'  outputs: {stack_output.name}, {mip_output.name}')

planned_paths = validate_unique_output_paths([item['group'] for item in prepared], dataset_root)
print(f'Preflight passed: {len(prepared)} group(s), {len(planned_paths)} unique output path(s).')

## 4. Reconstruction function
Existing complete output pairs are skipped unless `overwrite_outputs=True`; partial pairs fail before reconstruction.

In [ ]:
def process_fov(item):
    group = item['group']
    stack_output, mip_output = item['stack_output'], item['mip_output']
    existing = [path for path in (stack_output, mip_output) if path.exists()]
    if existing and not overwrite_outputs:
        if len(existing) == 2:
            return {**item, 'status': 'skipped_existing', 'runtime_s': 0.0}
        raise FileExistsError(f'Partial output pair exists: {existing}')

    params = SACDParams(
        pixel_nm=item['pixel_nm'], wavelength_nm=wavelength_nm, na=item['na'], mag=mag,
        iter1=iter1, iter2=iter2, ac_order=ac_order, subfactor=subfactor, frames_per_sacd=None,
        ifbackground=ifbackground, backgroundfactor=backgroundfactor, ifregistration=ifregistration,
        ifsparsedecon=ifsparsedecon, fidelity=fidelity, tcontinuity=tcontinuity, sparsity=sparsity,
        sparse_iterations=sparse_iterations,
    )
    started = perf_counter()
    total = len(group.files)
    progress_context = nullcontext(None)
    if show_progress:
        progress_context = Progress(SpinnerColumn(), TextColumn('[progress.description]{task.description}'), BarColumn(), MofNCompleteColumn(), TimeElapsedColumn(), TimeRemainingColumn())
    sacd_times = []
    with progress_context as progress:
        task_id = progress.add_task(item['fov_folder'].name, total=total) if progress is not None else None
        for time_index in group.time_indices:
            sacd_z = []
            for z_index in group.z_indices:
                input_file = group.files[(time_index, z_index)]
                if progress is not None:
                    progress.update(task_id, description=f'{item["fov_folder"].name} t{time_index} z{z_index}')
                raw = read_tiff_stack(input_file)
                sacd = reconstruct(raw, params)
                if sacd.ndim != 2:
                    raise ValueError(f'Expected one 2D SACD image from {input_file}, got {sacd.shape}')
                sacd_z.append(sacd)
                if progress is not None:
                    progress.advance(task_id)
            sacd_times.append(np.stack(sacd_z, axis=0))
    sacd_tzyx = np.stack(sacd_times, axis=0).astype(np.float32, copy=False)
    sacd_mip_tyx = np.max(sacd_tzyx, axis=1).astype(np.float32, copy=False)
    if not np.array_equal(sacd_mip_tyx, np.max(sacd_tzyx, axis=1)):
        raise AssertionError('SACD MIP validation failed')
    output_pixel_um = item['pixel_nm'] / 1000.0 / mag
    write_timelapse_tiff(stack_output, sacd_tzyx, axes='TZYX', pixel_size_um=output_pixel_um, z_spacing_um=item['z_spacing_um'], time_interval_s=item['time_interval_s'])
    write_timelapse_tiff(mip_output, sacd_mip_tyx, axes='TYX', pixel_size_um=output_pixel_um, time_interval_s=item['time_interval_s'])
    return {**item, 'status': 'written', 'runtime_s': perf_counter() - started, 'input_shape': raw.shape, 'stack_shape': sacd_tzyx.shape, 'mip_shape': sacd_mip_tyx.shape}

def validate_outputs(result):
    with tifffile.TiffFile(result['stack_output']) as tif:
        stack_shape, stack_axes, stack_dtype = tif.series[0].shape, tif.series[0].axes, tif.series[0].dtype
    with tifffile.TiffFile(result['mip_output']) as tif:
        mip_shape, mip_axes, mip_dtype = tif.series[0].shape, tif.series[0].axes, tif.series[0].dtype
    if stack_axes != 'TZYX' or mip_axes != 'TYX' or stack_dtype != np.dtype('float32') or mip_dtype != np.dtype('float32'):
        raise AssertionError(f'Unexpected output metadata: {stack_axes}/{stack_dtype}, {mip_axes}/{mip_dtype}')
    stack = tifffile.imread(result['stack_output'])
    mip = tifffile.imread(result['mip_output'])
    if not np.array_equal(mip, np.max(stack, axis=1)):
        raise AssertionError('Saved MIP does not equal max(saved TZYX, axis=1)')
    return stack_shape, mip_shape

## 5. Pilot the first FOV

In [ ]:
pilot_result = process_fov(prepared[0])
pilot_shapes = validate_outputs(pilot_result)
print(f"Pilot {pilot_result['status']}: {pilot_result['fov_folder'].name}, shapes={pilot_shapes}, runtime_s={pilot_result['runtime_s']:.1f}")

## 6. Pilot preview
Inspect the reconstructed z planes and their MIP before running the remaining FOVs.

In [ ]:
pilot_stack = tifffile.imread(pilot_result['stack_output'])
pilot_mip = tifffile.imread(pilot_result['mip_output'])
fig, axes = plt.subplots(1, pilot_stack.shape[1] + 1, figsize=(4 * (pilot_stack.shape[1] + 1), 4), squeeze=False)
for z_index, ax in enumerate(axes.ravel()[:-1]):
    ax.imshow(pilot_stack[0, z_index], cmap='hot')
    ax.set_title(f't0 z{z_index}')
    ax.axis('off')
axes.ravel()[-1].imshow(pilot_mip[0], cmap='hot')
axes.ravel()[-1].set_title('t0 SACD MIP')
axes.ravel()[-1].axis('off')
plt.tight_layout();

## 7. Full batch
Run after the pilot passes visual inspection. The pilot is safely reported as an existing output pair.

In [ ]:
results = [pilot_result]
if run_full_batch:
    for item in prepared[1:]:
        result = process_fov(item)
        validate_outputs(result)
        results.append(result)
        print(f"{result['status']}: {result['fov_folder'].name}, runtime_s={result['runtime_s']:.1f}")
else:
    print('Pilot complete. Set SACDPY_RUN_FULL_BATCH=1 when launching the notebook to process the remaining FOVs.')

## 8. Processing summary

In [ ]:
for result in results:
    print({
        'source_folder': str(result['fov_folder']), 'included_time_indices': result['group'].time_indices,
        'dropped_time_indices': result['dropped_time_indices'], 'input_shape': result.get('input_shape'),
        'stack_shape': result.get('stack_shape'), 'mip_shape': result.get('mip_shape'),
        'pixel_nm': result['pixel_nm'], 'na': result['na'], 'z_spacing_um': result['z_spacing_um'],
        'time_interval_s': result['time_interval_s'], 'stack_output': str(result['stack_output']),
        'mip_output': str(result['mip_output']), 'runtime_s': result['runtime_s'], 'status': result['status'],
    })